# R4 · 1 · Detector ampliado

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 a 3):
1. suba **`r4_lib.py`** (pasta `data_imdc_2026/RODADA4/`) e confira que `r3_lib.py` e `r2_lib.py` estão lá;
2. as pastas `resultados_uf_r4`, `revisoes_r4`, `cache_r4` e `avaliacao_r4` são criadas sozinhas.

**Usa a rodada 3:** o normal (ENS_grafos3_λ) vem de `cache_r3/normal_ens_grafos3_bb_clim.parquet` e as misturas da R3 entram na comparação.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r4/uf/`; as revisões de julho a outubro, para `revisoes_r4/uf/`; as tabelas, para `avaliacao_r4/` (com data e hora no nome e uma cópia `_ultimo`).

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (mar, sorotipo por ano, variáveis, detector por método × corte, parâmetros, um arquivo por modelo × temporada, os pesos das redes a cada 5 épocas, um PNG por UF) e o que já existe é pulado.

CPU comum, 20–40 min. Rode depois do R4_0 (se for usar o sorotipo).

**O que mudou em relação ao detector da R3:** o alvo é o mesmo (log do tamanho da temporada relativo ao normal da UF). Entram variáveis novas:

| Grupo | Variáveis | Fonte |
|---|---|---|
| Estabilidade entre surtos e alerta precoce | autocorrelação, variância e as tendências de ambas nas últimas 26 semanas; dispersão; piso relativo | Rypdal 2019; Brett 2020 |
| Imunidade da população | tamanho relativo das últimas 6 temporadas com esquecimento (τ = 1 e 3 temporadas); ataque acumulado em 3 temporadas | McGough 2021 |
| Seca e clima observado | índice de seca tipo SPEI (chuva − evapotranspiração, 13 e 26 semanas); seca máxima no último ano; anomalias de temperatura, chuva e umidade | Lowe 2021 |
| Clima previsto | SEAS5 emitida até o corte: anomalias de temperatura, chuva e umidade para 1–3 e 4–6 meses à frente | ECMWF (já no projeto) |
| Mar | Índico (IOBW), Atlântico tropical norte e sul, Atlântico sudoeste e Niño 3.4 (ERSST v5): média de 3 meses e variação em 6 | Chen 2024; Beal 2025 |
| Sorotipo (se o R4_0 rodou) | fração de cada sorotipo, sorotipo emergente, sorotipos antes raros | Finch 2025 |

**Métodos:** os 3 da R3, mais **duas etapas** (primeiro P(ano grande), depois o tamanho dado grande ou não; Francisco 2024), mais a média. Cada um também ganha uma versão **conforme** (`+conf`), em que cada quantil é corrigido pelos erros fora da amostra das temporadas já terminadas. Isso alonga a cauda de cima quando o detector vinha errando os anos grandes para baixo.

**Escolha, por corte:** fica o de menor pinball nas temporadas ≤ 2021, com os **anos grandes valendo 3×**. Só a afinação (2018–2021) tinha poucos anos grandes e premiava o LightGBM, que não extrapola: no teste de fumaça ele previu SC 2023 em ×9, quando o real foi ×206.

**Sem vazamento:**
- os sinais são escolhidos só com temporadas ≤ 2021 (as variáveis nacionais, iguais em todas as UFs, precisam de correlação maior: |ρ| ≥ 0,30);
- o treino é em janela expansiva;
- as anomalias de clima e do mar usam só anos anteriores;
- o mar usa o mês anterior ao do corte (o último publicado).


In [ ]:
NOTEBOOK = 'r4_detector'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r4', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'r4_lib.py', 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    subprocess.run('pip install -q netCDF4', shell=True)
sys.path.insert(0, PASTA_LOCAL)
PASTA_R4 = os.path.join(PASTA_DRIVE, 'resultados_uf_r4')
PASTA_REV4 = os.path.join(PASTA_DRIVE, 'revisoes_r4')
PASTA_C4 = os.path.join(PASTA_DRIVE, 'cache_r4')
PASTA_AV4 = os.path.join(PASTA_DRIVE, 'avaliacao_r4')
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R4, 'uf'), os.path.join(PASTA_REV4, 'uf'), PASTA_C4, os.path.join(PASTA_C4, 'ckpt'), PASTA_AV4):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, r4_lib as R4
for m in (V, L, CM, U, S, R2, R3, R4):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
R4.PESO_ABS = 0.5     # critério misto = (1 − p)·R × referência + p·WIS absoluto relativo. Suba (ex.: 0.7) para priorizar a altura dos picos


In [ ]:
# @title 2 · Temperatura do mar (ERSST v5, NOAA; ~150 MB uma vez) e sorotipo (se houver)
try:
    OC = R4.indices_oceano(PASTA_C4)
except Exception:
    traceback.print_exc(); OC = None
SOR = R4.ler_sorotipo(PASTA_C4)
arq_ext = os.path.join(PASTA_C4, 'externos_r4.csv')
EXT = pd.read_csv(arq_ext) if os.path.exists(arq_ext) else None
print('mar:', None if OC is None else f'{OC.mes.min().date()} → {OC.mes.max().date()}',
      '· sorotipo:', None if SOR is None else f'{len(SOR)} linhas UF × semana', '· externos:', None if EXT is None else list(EXT.columns))
if OC is not None:
    display(OC.tail(8).round(2))


In [ ]:
# @title 3 · Variáveis e alvo (cache; para refazer, apague cache_r4/detector4_* e cache_r4/sinais4.json)
X = R4.tabela_detector4(ctx, list(range(2011, 2026)), PASTA_C4, OC, SOR, EXT)
NOVAS = [c for c in X.columns if c.startswith(('ews_', 'piso', 'imun', 'ataque', 'anom_s1', 'seca', 'clim_', 'seas_', 'sor_'))
         or c.split('_')[0] in R4.REGIOES_SST]
print(X.shape, '· variáveis novas:', len(NOVAS))
X[X.corte == 'jun'][NOVAS].describe().T.round(2)


In [ ]:
# @title 4 · Sinais escolhidos só com temporadas ≤ 2021
ARQ_SIN = os.path.join(PASTA_C4, 'sinais4.json')
SINAIS = {k: R4.selecionar_sinais4(X, k, ate_temporada=max(TUNING), arq=ARQ_SIN) for k in R4.CORTES}
t = pd.DataFrame({k: pd.Series(v['rho']) for k, v in R4.json_le(ARQ_SIN).items()}).round(2)
t['nova'] = t.index.isin(NOVAS)
t.sort_values('jun', key=abs, ascending=False)


In [ ]:
# @title 5 · Detector (janela expansiva) + conformes, avaliação e escolha por corte
P = R4.prever_detector4(ctx, X, SINAIS, list(range(2014, 2026)), PASTA_C4)
P.to_parquet(os.path.join(PASTA_C4, 'detector4_todos.parquet'))
TD = R4.avaliar_detector4(P)
PE = R4.pinball_escolha(P)
TD = TD.merge(PE, on=['metodo', 'corte'], how='left')
R4.salvar_tabela(TD, PASTA_AV4, 'detector4_avaliacao')
ESC = R4.escolher_detector4(PE)
R4.json_grava(ESC, os.path.join(PASTA_C4, 'detector4_escolhido.json'))
print('escolhido por corte (menor pinball nas temporadas ≤ 2021, anos grandes valendo 3×):', ESC)
display(PE.pivot_table(index='metodo', columns='corte', values='pinball_escolha').round(3))
# comparação com o detector escolhido na R3 (mesmas métricas)
E3 = R3.json_le(os.path.join(PASTA_C3, 'detector_escolhido.json'), {})
P3 = pd.concat([pd.read_parquet(os.path.join(PASTA_C3, f'detector_{m}_{k}.parquet')).assign(metodo=f'R3:{m}') for k, m in E3.items()]) if E3 else None
T3 = R4.avaliar_detector4(P3) if P3 is not None else None
esc4 = pd.concat([TD[(TD.metodo == m) & (TD.corte == k)].assign(metodo=f'R4:{m}') for k, m in ESC.items()])
CMP = pd.concat([T3, esc4]) if T3 is not None else esc4
R4.salvar_tabela(CMP, PASTA_AV4, 'detector_r3_x_r4')
CMP[CMP.grupo != 'anterior'].pivot_table(index=['corte', 'metodo'], columns='grupo',
    values=['pinball', 'corr_mediana', 'IC90', 'sensibilidade', 'precisao', 'vies_grandes_x']).round(2)


In [ ]:
# @title 6 · O que o detector disse em junho de cada temporada de teste (× o normal da UF)
_, DET = R4.detector_escolhido4(PASTA_C4)
DJ = DET[(DET.corte == 'jun') & DET.season.isin(TESTE)].copy()
DJ['uf'] = DJ.uf_code.map(R2.SIGLAS)
DJ['real_x'], DJ['previsto_x'], DJ['q95_x'] = np.exp(DJ.alvo), np.exp(DJ.pred), np.exp(DJ.upper_90)
TAB = DJ.pivot_table(index='uf', columns='season', values=['real_x', 'previsto_x', 'q95_x', 'p_grande']).round(2)
R4.salvar_tabela(TAB.reset_index(), PASTA_AV4, 'detector4_junho_por_uf')
TAB
